# 06 时序差分 (TD)：SARSA 与 Q-learning

**TD = MC 的样本效率 + DP 的自举（bootstrap）**

TD 不等回合结束就更新：

$$V(s_t) \leftarrow V(s_t) + \alpha \underbrace{[r_{t+1} + \gamma V(s_{t+1}) - V(s_t)]}_{\text{TD error } \delta_t}$$

## SARSA vs Q-learning（**经典面试题**）

| | SARSA (on-policy) | Q-learning (off-policy) |
|--|------------------|----------------------|
| Update target | $r + \gamma Q(s', \mathbf{a'})$ where $a' \sim \pi$ | $r + \gamma \max_{a'} Q(s', a')$ |
| 直觉 | "学我实际会做的策略" | "学最优策略，无论我现在用什么" |
| 风险偏好 | 保守（考虑探索的代价） | 激进（认为永远走最优） |
| 经典案例 | Cliff Walking 中 SARSA 走稳路 | Q-learning 走悬崖边最优路 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

ROWS, COLS = 4, 12
ACTIONS = ['U', 'D', 'L', 'R']
DELTAS = {'U':(-1,0),'D':(1,0),'L':(0,-1),'R':(0,1)}
START = (3, 0)
GOAL = (3, 11)
CLIFF = [(3, c) for c in range(1, 11)]

def step(state, action):
    r, c = state
    dr, dc = DELTAS[action]
    nr, nc = max(0, min(ROWS-1, r+dr)), max(0, min(COLS-1, c+dc))
    next_state = (nr, nc)
    if next_state in CLIFF:
        return START, -100, False
    if next_state == GOAL:
        return next_state, -1, True
    return next_state, -1, False

def epsilon_greedy(Q, s, eps=0.1):
    if np.random.rand() < eps:
        return np.random.randint(4)
    return int(np.argmax(Q[s]))

## 1. SARSA 实现

$$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s', a') - Q(s,a)]$$

其中 $a'$ 也是按 ε-greedy 选的（**on-policy**）。

In [ ]:
def sarsa(num_episodes=500, alpha=0.5, gamma=1.0, eps=0.1):
    Q = np.zeros((ROWS, COLS, 4))
    rewards_per_ep = []
    for ep in range(num_episodes):
        s = START
        a = epsilon_greedy(Q[s], s, eps)
        total_r = 0
        while True:
            s_next, r, done = step(s, ACTIONS[a])
            total_r += r
            a_next = epsilon_greedy(Q[s_next], s_next, eps)
            Q[s][a] += alpha * (r + gamma * Q[s_next][a_next] - Q[s][a])
            s, a = s_next, a_next
            if done:
                break
        rewards_per_ep.append(total_r)
    return Q, rewards_per_ep

## 2. Q-learning 实现

$$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s', a') - Q(s,a)]$$

target 用 max（**off-policy**：估计的是 greedy 策略，但行为是 ε-greedy）。

In [ ]:
def q_learning(num_episodes=500, alpha=0.5, gamma=1.0, eps=0.1):
    Q = np.zeros((ROWS, COLS, 4))
    rewards_per_ep = []
    for ep in range(num_episodes):
        s = START
        total_r = 0
        while True:
            a = epsilon_greedy(Q[s], s, eps)
            s_next, r, done = step(s, ACTIONS[a])
            total_r += r
            target = r + gamma * np.max(Q[s_next])
            Q[s][a] += alpha * (target - Q[s][a])
            s = s_next
            if done:
                break
        rewards_per_ep.append(total_r)
    return Q, rewards_per_ep

Q_sarsa, r_sarsa = sarsa()
Q_qlearn, r_qlearn = q_learning()

def smooth(x, w=20):
    x = np.array(x, dtype=float)
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(smooth(r_sarsa), label='SARSA')
plt.plot(smooth(r_qlearn), label='Q-learning')
plt.xlabel('Episode'); plt.ylabel('Total Reward (smoothed)')
plt.title('Cliff Walking: SARSA vs Q-learning')
plt.legend(); plt.grid(True); plt.show()

**观察**：
- Q-learning 学到的最优策略是「沿悬崖走」（最短），但因 ε-greedy 偶尔跌落，平均奖励低
- SARSA 考虑了探索代价，学到「走上面绕路」，平均奖励高

**给定位算法的启示**：
- 如果你的策略部署时不再探索（greedy），用 Q-learning
- 如果策略会带噪声（如随机扰动 GPS 修正量），用 SARSA 更稳

## 3. 可视化两种策略

In [ ]:
def plot_policy(Q, title):
    fig, ax = plt.subplots(figsize=(10, 3))
    grid = np.zeros((ROWS, COLS))
    for c in range(1, 11):
        grid[3, c] = -1
    grid[GOAL] = 1
    ax.imshow(grid, cmap='RdYlGn')
    arrows = {'U':(0,0.3),'D':(0,-0.3),'L':(-0.3,0),'R':(0.3,0)}
    for r in range(ROWS):
        for c in range(COLS):
            if (r,c) in CLIFF or (r,c) == GOAL:
                continue
            best = ACTIONS[np.argmax(Q[r,c])]
            dx, dy = arrows[best]
            ax.arrow(c, r, dx, -dy, head_width=0.15, color='black')
    ax.set_title(title)
    ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS))
    plt.show()

plot_policy(Q_sarsa, 'SARSA Policy (safer)')
plot_policy(Q_qlearn, 'Q-learning Policy (optimal but risky)')

## 4. 总结

至此，你已掌握：
- 表格 Q-learning（这是 DQN 的基础）
- on-policy vs off-policy（关系到第 3 章 PPO 与第 2 章 DQN 的本质区别）
- 探索 vs 利用（ε-greedy 是最简单的方案）

**下一步**：进入 `02_value_based/` —— 用神经网络替代 Q 表，进入 DQN 时代。